In [1]:
# 1. Import the existing src APIs and discover all result models.
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from evaluation import evaluate
from datahandler import DataLoader
from gnn import GCNNodeClassifier

RESULTS_DIR = SRC_DIR / "results"
SIMULATION_PATH = PROJECT_ROOT / "data" / "simulation_scores.csv"
result_paths = sorted(
    path for path in RESULTS_DIR.iterdir()
    if path.is_dir() and (path / "best_seed_predictions.csv").exists()
)

print(f"Discovered {len(result_paths)} result models:")
for path in result_paths:
    print(f"  {path.name}")

/Users/sujay/Downloads/DATA_P2/candidate_package/.venv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: Could not load this library: /Users/sujay/Downloads/DATA_P2/candidate_package/.venv/lib/python3.12/site-packages/pyg_lib/libpyg.so
  import torch_geometric.typing
/Users/sujay/Downloads/DATA_P2/candidate_package/.venv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /Users/sujay/Downloads/DATA_P2/candidate_package/.venv/lib/python3.12/site-packages/torch_scatter/_scatter_cpu.so
  import torch_geometric.typing
/Users/sujay/Downloads/DATA_P2/candidate_package/.venv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: /Use

Discovered 8 result models:
  baseline_multimodal
  baseline_only_a
  baseline_only_b
  gcn_multimodal_identity
  gcn_multimodal_real
  gcn_multimodal_shuffled
  gcn_only_a
  gcn_only_b


In [16]:
# 2. Load every model's saved predictions and merge simulation scores once.
simulation_scores = pd.read_csv(SIMULATION_PATH)
model_predictions = {}

for result_path in result_paths:
    predictions = pd.read_csv(result_path / "best_seed_predictions.csv")
    merged = predictions.merge(
        simulation_scores,
        on=["subject_id", "node_id"],
        how="inner",
        validate="one_to_one",
    )
    model_predictions[result_path.name] = merged

print("Loaded prediction tables:")
for model_name, predictions in model_predictions.items():
    test_rows = predictions[predictions["split"] == "test"]
    print(f"  {model_name}: {len(predictions)} rows, {len(test_rows)} test rows")

Loaded prediction tables:
  baseline_multimodal: 9520 rows, 1904 test rows
  baseline_only_a: 9520 rows, 1904 test rows
  baseline_only_b: 9520 rows, 1904 test rows
  gcn_multimodal_identity: 9520 rows, 1904 test rows
  gcn_multimodal_real: 9520 rows, 1904 test rows
  gcn_multimodal_shuffled: 9520 rows, 1904 test rows
  gcn_only_a: 9520 rows, 1904 test rows
  gcn_only_b: 9520 rows, 1904 test rows


In [7]:
# Standalone simulation-score evaluation for every model and every split.
simulation_evaluation_frames = {}
simulation_evaluation_results = []

for model_name, predictions in model_predictions.items():
    for split_name in ["train", "val", "test"]:
        split_predictions = predictions[predictions["split"] == split_name].copy()

        simulation_df = split_predictions[
            ["subject_id", "node_id", "node_labels", "sim_score"]
        ].rename(columns={"sim_score": "model_score"})
        simulation_evaluation_frames[(model_name, split_name)] = simulation_df

        result = evaluate(simulation_df.copy())
        simulation_evaluation_results.append({
            "model": model_name,
            "split": split_name,
            **result,
        })

simulation_metrics_table = (
    pd.DataFrame(simulation_evaluation_results)
    .set_index(["model", "split"])
    .sort_index()
)

print("Standalone simulation evaluation for all models and splits:")
print(simulation_metrics_table.round(4))
print("\\nRows evaluated by model and split:")
print({key: len(frame) for key, frame in simulation_evaluation_frames.items()})

Standalone simulation evaluation for all models and splits:
                                auprc   auroc  top_k_dice  prevalence
model                   split                                        
baseline_multimodal     test   0.4048  0.7726      0.4563      0.0720
                        train  0.4147  0.7593      0.4970      0.0734
                        val    0.4371  0.7722      0.5548      0.0714
baseline_only_a         test   0.4048  0.7726      0.4563      0.0720
                        train  0.4147  0.7593      0.4970      0.0734
                        val    0.4371  0.7722      0.5548      0.0714
baseline_only_b         test   0.4048  0.7726      0.4563      0.0720
                        train  0.4147  0.7593      0.4970      0.0734
                        val    0.4371  0.7722      0.5548      0.0714
gcn_multimodal_identity test   0.4048  0.7726      0.4563      0.0720
                        train  0.4147  0.7593      0.4970      0.0734
                        val   

In [8]:
split_predictions

,seed,split,subject_id,node_id,model_score,node_labels,sim_score,sim_confidence
7616,78,test,sub-002,0,0.082865,0,0.020047,0.2010
7617,78,test,sub-002,1,0.089401,0,0.045399,0.2010
7618,78,test,sub-002,2,0.055655,0,0.015383,0.2010
7619,78,test,sub-002,3,0.052676,0,0.057124,0.2010
7620,78,test,sub-002,4,0.044189,0,0.062150,0.2010
...,...,...,...,...,...,...,...,...
9515,78,test,sub-135,63,0.050178,0,0.060440,0.2109
9516,78,test,sub-135,64,0.155631,0,0.000000,0.2109
9517,78,test,sub-135,65,0.113052,0,0.011894,0.2109
9518,78,test,sub-135,66,0.128876,0,0.017934,0.2109


In [ ]:
# Inspect each model's saved evaluation summary without overwriting predictions.
evaluation_summaries = {}

for result_path in result_paths:
    summary_path = result_path / "evaluation_summary.csv"
    if summary_path.exists():
        evaluation_summaries[result_path.name] = pd.read_csv(summary_path)
        print(f"{result_path.name}")
        display(evaluation_summaries[result_path.name])

   split  auprc_mean  auprc_std  auroc_mean  auroc_std  top_k_dice_mean  \
0   test    0.277342   0.049884    0.777746   0.011498         0.334711   
1  train    0.385838   0.076188    0.854501   0.025533         0.417194   
2    val    0.255755   0.063724    0.765725   0.015827         0.305289   

   top_k_dice_std  prevalence_mean  prevalence_std  
0        0.039084         0.071954             0.0  
1        0.042371         0.073354             0.0  
2        0.053030         0.071429             0.0  
   split  auprc_mean  auprc_std  auroc_mean  auroc_std  top_k_dice_mean  \
0   test    0.232718   0.045575    0.757659   0.021801         0.296701   
1  train    0.295701   0.053951    0.822713   0.023710         0.345567   
2    val    0.190701   0.014514    0.733445   0.015021         0.238384   

   top_k_dice_std  prevalence_mean  prevalence_std  
0        0.047822         0.071954             0.0  
1        0.044580         0.073354             0.0  
2        0.046728         0

In [17]:
# Correlation between model_score and sim_score for every split.
from scipy.stats import pearsonr, spearmanr

correlation_rows = []
for model_name, predictions in model_predictions.items():
    for split_name in ["train", "val", "test"]:
        split_predictions = predictions[predictions["split"] == split_name]
        model_scores = split_predictions["model_score"].to_numpy()
        simulation_scores_for_split = split_predictions["sim_score"].to_numpy()

        correlation_rows.append({
            "model": model_name,
            "split": split_name,
            "rows": len(split_predictions),
            "pearson_correlation": pearsonr(
                model_scores, simulation_scores_for_split
            ).statistic,
            "spearman_correlation": spearmanr(
                model_scores, simulation_scores_for_split
            ).statistic,
        })

correlation_table = (
    pd.DataFrame(correlation_rows)
    .set_index(["model", "split"])
    .sort_index()
)

print("Correlation between model_score and sim_score:")
print(correlation_table.round(4))

Correlation between model_score and sim_score:
                               rows  pearson_correlation  spearman_correlation
model                   split                                                 
baseline_multimodal     test   1904               0.2242                0.1531
                        train  5712               0.3121                0.1721
                        val    1904               0.2580                0.1708
baseline_only_a         test   1904               0.1799                0.1323
                        train  5712               0.2205                0.1406
                        val    1904               0.1457                0.1183
baseline_only_b         test   1904               0.1362                0.1488
                        train  5712               0.2618                0.1665
                        val    1904               0.1853                0.1692
gcn_multimodal_identity test   1904               0.2719                0.1255
     

In [20]:
# Where model_score and sim_score disagree the most.
disagreement_rows = []

for model_name, predictions in model_predictions.items():
    for split_name in ["train", "val", "test"]:
        split_df = predictions[predictions["split"] == split_name].copy()
        split_df["absolute_difference"] = (
            split_df["model_score"] - split_df["sim_score"]
        ).abs()
        split_df["model"] = model_name
        split_df["split_name"] = split_name
        disagreement_rows.append(split_df[
            [
                "model",
                "split_name",
                "subject_id",
                "node_id",
                "model_score",
                "sim_score",
                "absolute_difference",
            ]
        ])

disagreement_table = pd.concat(disagreement_rows, ignore_index=True)
most_discordant = disagreement_table.sort_values(
    "absolute_difference", ascending=False
).head(20)

print("Top 20 disagreements between model_score and sim_score:")
print(most_discordant.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

Top 20 disagreements between model_score and sim_score:
                  model split_name subject_id  node_id  model_score  sim_score  absolute_difference
        baseline_only_b      train    sub-129       34       0.0000     1.0000               1.0000
        baseline_only_b       test    sub-081       51       0.0001     1.0000               0.9999
        baseline_only_b        val    sub-033       66       0.0003     1.0000               0.9997
    baseline_multimodal      train    sub-129       34       0.0003     1.0000               0.9997
        baseline_only_a       test    sub-027       32       0.0004     1.0000               0.9996
        baseline_only_b      train    sub-090       19       0.0007     1.0000               0.9993
        baseline_only_b        val    sub-023       64       0.0008     1.0000               0.9992
    baseline_multimodal       test    sub-027       32       0.0009     1.0000               0.9991
    baseline_multimodal        val    sub-02

In [24]:
# Evaluate the naive average of model_score and sim_score.
average_evaluation_results = []
average_evaluation_frames = {}

for model_name, predictions in model_predictions.items():
    for split_name in ["train", "val", "test"]:
        split_predictions = predictions[predictions["split"] == split_name].copy()

        average_df = split_predictions[
            ["subject_id", "node_id", "node_labels", "model_score", "sim_score"]
        ].copy()
        average_df["model_score"] = (
            average_df["model_score"] + average_df["sim_score"]
        ) / 2
        average_evaluation_frames[(model_name, split_name)] = average_df

        result = evaluate(average_df.copy())
        average_evaluation_results.append({
            "model": model_name,
            "split": split_name,
            **result,
        })

average_metrics_table = (
    pd.DataFrame(average_evaluation_results)
    .set_index(["model", "split"])
    .sort_index()
)

print("Naive average evaluation: (model_score + sim_score) / 2")
print(average_metrics_table.round(4))
print("\nRows evaluated:")
print({key: len(frame) for key, frame in average_evaluation_frames.items()})

Naive average evaluation: (model_score + sim_score) / 2
                                auprc   auroc  top_k_dice  prevalence
model                   split                                        
baseline_multimodal     test   0.6041  0.8763      0.5548      0.0720
                        train  0.6778  0.9197      0.6202      0.0734
                        val    0.6071  0.8564      0.6248      0.0714
baseline_only_a         test   0.5682  0.8631      0.5106      0.0720
                        train  0.6184  0.9020      0.5880      0.0734
                        val    0.5806  0.8540      0.6021      0.0714
baseline_only_b         test   0.5242  0.8118      0.5343      0.0720
                        train  0.6477  0.9129      0.6127      0.0734
                        val    0.5434  0.8164      0.5736      0.0714
gcn_multimodal_identity test   0.6502  0.9108      0.5572      0.0720
                        train  0.6640  0.9055      0.6189      0.0734
                        val    0.6

In [11]:
# Summary tables: mean +/- standard deviation across seeds for every model.
summary_frames = []

for result_path in result_paths:
    metrics_path = result_path / "evaluation_by_seed.csv"
    if not metrics_path.exists():
        continue

    metrics = pd.read_csv(metrics_path)
    metrics["model"] = result_path.name
    summary_frames.append(metrics)

all_seed_metrics = pd.concat(summary_frames, ignore_index=True)

model_order = [
    path.name for path in result_paths
    if path.name in all_seed_metrics["model"].unique()
]

summary = (
    all_seed_metrics
    .groupby(["model", "split"], sort=False)[
        ["auprc", "auroc", "top_k_dice"]
    ]
    .agg(["mean", "std"])
    .reset_index()
)

summary["model"] = pd.Categorical(
    summary["model"], categories=model_order, ordered=True
)
summary = summary.sort_values(["split", "model"])

for metric in ["auprc", "auroc", "top_k_dice"]:
    summary[(metric, "mean_std")] = summary.apply(
        lambda row: f"{row[(metric, 'mean')]:.4f} +/- {row[(metric, 'std')]:.4f}",
        axis=1,
    )

summary_tables = {}
for split_name, output_name in [
    ("train", "train_df"),
    ("val", "validation_df"),
    ("test", "test_df"),
]:
    split_table = summary[summary["split"] == split_name].copy()
    split_table.columns = [
        "_".join(column).strip("_")
        if isinstance(column, tuple)
        else column
        for column in split_table.columns
    ]
    summary_tables[output_name] = split_table[
        ["model", "auprc_mean_std", "auroc_mean_std", "top_k_dice_mean_std"]
    ].rename(
        columns={
            "auprc_mean_std": "AUPRC (mean +/- std)",
            "auroc_mean_std": "AUROC (mean +/- std)",
            "top_k_dice_mean_std": "DICE (mean +/- std)",
        }
    ).reset_index(drop=True)

train_df = summary_tables["train_df"]
validation_df = summary_tables["validation_df"]
test_df = summary_tables["test_df"]

print("TRAIN")
display(train_df)
print("VALIDATION")
display(validation_df)
print("TEST")
display(test_df)

TRAIN


,model,AUPRC (mean +/- std),AUROC (mean +/- std),DICE (mean +/- std)
0,baseline_multimodal,0.3858 +/- 0.0762,0.8545 +/- 0.0255,0.4172 +/- 0.0424
1,baseline_only_a,0.2957 +/- 0.0540,0.8227 +/- 0.0237,0.3456 +/- 0.0446
2,baseline_only_b,0.3149 +/- 0.0812,0.8037 +/- 0.0359,0.3439 +/- 0.0602
3,gcn_multimodal_identity,0.4295 +/- 0.0127,0.8492 +/- 0.0067,0.4485 +/- 0.0149
4,gcn_multimodal_real,0.2001 +/- 0.0121,0.7305 +/- 0.0101,0.2454 +/- 0.0261
5,gcn_multimodal_shuffled,0.1981 +/- 0.0122,0.7293 +/- 0.0104,0.2404 +/- 0.0248
6,gcn_only_a,0.1734 +/- 0.0026,0.7003 +/- 0.0032,0.1942 +/- 0.0021
7,gcn_only_b,0.1215 +/- 0.0233,0.6118 +/- 0.0338,0.1465 +/- 0.0472


VALIDATION


,model,AUPRC (mean +/- std),AUROC (mean +/- std),DICE (mean +/- std)
0,baseline_multimodal,0.2558 +/- 0.0637,0.7657 +/- 0.0158,0.3053 +/- 0.0530
1,baseline_only_a,0.1907 +/- 0.0145,0.7334 +/- 0.0150,0.2384 +/- 0.0467
2,baseline_only_b,0.1926 +/- 0.0436,0.6893 +/- 0.0099,0.2302 +/- 0.0427
3,gcn_multimodal_identity,0.3698 +/- 0.0115,0.8092 +/- 0.0043,0.4011 +/- 0.0132
4,gcn_multimodal_real,0.1729 +/- 0.0073,0.6607 +/- 0.0103,0.2035 +/- 0.0156
5,gcn_multimodal_shuffled,0.1714 +/- 0.0086,0.6599 +/- 0.0119,0.2021 +/- 0.0134
6,gcn_only_a,0.1574 +/- 0.0051,0.6427 +/- 0.0051,0.1721 +/- 0.0039
7,gcn_only_b,0.1040 +/- 0.0196,0.5630 +/- 0.0279,0.1260 +/- 0.0306


TEST


,model,AUPRC (mean +/- std),AUROC (mean +/- std),DICE (mean +/- std)
0,baseline_multimodal,0.2773 +/- 0.0499,0.7777 +/- 0.0115,0.3347 +/- 0.0391
1,baseline_only_a,0.2327 +/- 0.0456,0.7577 +/- 0.0218,0.2967 +/- 0.0478
2,baseline_only_b,0.1966 +/- 0.0244,0.6866 +/- 0.0231,0.2382 +/- 0.0398
3,gcn_multimodal_identity,0.4353 +/- 0.0125,0.8598 +/- 0.0027,0.4563 +/- 0.0094
4,gcn_multimodal_real,0.1506 +/- 0.0104,0.7044 +/- 0.0089,0.2031 +/- 0.0169
5,gcn_multimodal_shuffled,0.1488 +/- 0.0104,0.7032 +/- 0.0090,0.2009 +/- 0.0190
6,gcn_only_a,0.1306 +/- 0.0025,0.6789 +/- 0.0031,0.1834 +/- 0.0073
7,gcn_only_b,0.1101 +/- 0.0192,0.5994 +/- 0.0333,0.1067 +/- 0.0234


In [23]:
# Count highly discordant model/simulation pairs.
DISCORDANCE_LOWER = 0.8
DISCORDANCE_UPPER = 1.0

count_source = disagreement_table.copy()
count_source["is_highly_discordant"] = count_source[
    "absolute_difference"
].between(DISCORDANCE_LOWER, DISCORDANCE_UPPER, inclusive="both")

discordant_pair_counts = (
    count_source
    .groupby(["model", "split_name"], sort=True)
    .agg(
        discordant_pairs=("is_highly_discordant", "sum"),
        total_pairs=("is_highly_discordant", "size"),
    )
    .reset_index()
)
discordant_pair_counts["discordant_percent"] = (
    100 * discordant_pair_counts["discordant_pairs"]
    / discordant_pair_counts["total_pairs"]
)

print(
    f"Pairs with {DISCORDANCE_LOWER:.1f} <= abs(model_score - sim_score) "
    f"<= {DISCORDANCE_UPPER:.1f}:"
)
discordant_pair_counts

Pairs with 0.8 <= abs(model_score - sim_score) <= 1.0:


,model,split_name,discordant_pairs,total_pairs,discordant_percent
0,baseline_multimodal,test,59,1904,3.098739
1,baseline_multimodal,train,175,5712,3.063725
2,baseline_multimodal,val,65,1904,3.413866
3,baseline_only_a,test,50,1904,2.626050
4,baseline_only_a,train,161,5712,2.818627
5,baseline_only_a,val,59,1904,3.098739
6,baseline_only_b,test,66,1904,3.466387
7,baseline_only_b,train,178,5712,3.116246
8,baseline_only_b,val,64,1904,3.361345
9,gcn_multimodal_identity,test,46,1904,2.415966


In [ ]:
# predictions = pd.read_csv("/Users/sujay/Downloads/DATA_P2/candidate_package/src/results/baseline_multimodal/best_seed_predictions.csv")

# test_predictions = predictions[predictions["split"] == "test"]
# test_predictions["prob_abnormal"] = test_predictions["model_score"]
# test_predictions[["subject_id", "node_id", "prob_abnormal"]].to_csv("predictions.csv", index=False)